## Building a Simple Chatbot

In [1]:
import nltk
import numpy as np
import random
import string # to process standard python strings

In [6]:
f=open('/home/nate/NextCloud/Roam/Classes/NLP/week_11/chatbots.txt','r',errors = 'ignore')
raw=f.read()
raw=raw.lower()# converts to lowercase
nltk.download('punkt_tab') # first-time use only
nltk.download('wordnet') # first-time use only
sent_tokens = nltk.sent_tokenize(raw)# converts to list of sentences 
word_tokens = nltk.word_tokenize(raw)# converts to list of words

[nltk_data] Downloading package punkt_tab to /home/nate/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /home/nate/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [7]:
sent_tokens

['a chatbot is a software application used to conduct an on-line chat conversation via text or text-to-speech, in lieu of providing direct contact with a live human agent.',
 '[1] a chatbot is a type of software that can automate conversations and interact with people through messaging platforms.',
 '[2] designed to convincingly simulate the way a human would behave as a conversational partner, chatbot systems typically require continuous tuning and testing, and many in production remain unable to adequately converse or pass the industry standard turing test.',
 '[3] the term "chatterbot" was originally coined by michael mauldin (creator of the first verbot) in 1994 to describe these conversational programs.',
 '[4]\n\nchatbots are used in dialog systems for various purposes including customer service, request routing, or information gathering.',
 'while some chatbot applications use extensive word-classification processes, natural language processors, and sophisticated ai, others simp

In [8]:
lemmer = nltk.stem.WordNetLemmatizer()
#WordNet is a semantically-oriented dictionary of English included in NLTK.
def LemTokens(tokens):
    return [lemmer.lemmatize(token) for token in tokens]
remove_punct_dict = dict((ord(punct), None) for punct in string.punctuation)
def LemNormalize(text):
    return LemTokens(nltk.word_tokenize(text.lower().translate(remove_punct_dict)))

In [9]:
GREETING_INPUTS = ("hello", "hi", "greetings", "sup", "what's up","hey",)
GREETING_RESPONSES = ["hi", "hey", "*nods*", "hi there", "hello", "I am glad! You are talking to me"]
def greeting(sentence):
    for word in sentence.split():
        if word.lower() in GREETING_INPUTS:
            return random.choice(GREETING_RESPONSES)

In [10]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [11]:
def response(user_response):
    robo_response=''
    sent_tokens.append(user_response)
    TfidfVec = TfidfVectorizer(tokenizer=LemNormalize, stop_words='english')
    tfidf = TfidfVec.fit_transform(sent_tokens)
    vals = cosine_similarity(tfidf[-1], tfidf)
    idx=vals.argsort()[0][-2]
    flat = vals.flatten()
    flat.sort()
    req_tfidf = flat[-2]
    if(req_tfidf==0):
        robo_response=robo_response+"I am sorry! I don't understand you"
        return robo_response
    else:
        robo_response = robo_response+sent_tokens[idx]
        return robo_response

In [12]:
flag=True
print("Hi. I will answer your queries about Chatbots. To exit, type Bye!")
while(flag==True):
    user_response = input()
    user_response=user_response.lower()
    if(user_response!='bye'):
        if(user_response=='thanks' or user_response=='thank you' ):
            flag=False
            print("ROBO: You are welcome..")
        else:
            if(greeting(user_response)!=None):
                print("ROBO: "+greeting(user_response))
            else:
                print("ROBO: ",end="")
                print(response(user_response))
                sent_tokens.remove(user_response)
    else:
        flag=False
        print("ROBO: Bye! ")

Hi. I will answer your queries about Chatbots. To exit, type Bye!


 Tell me about weather patterns!


ROBO: 

/home/nate/venvs/nlp/lib/python3.13/site-packages/sklearn/feature_extraction/text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


by responding to any input that contains the word 'mother' with 'tell me more about your family').


/home/nate/venvs/nlp/lib/python3.13/site-packages/sklearn/feature_extraction/text.py:411: UserWarning: Your stop_words may be inconsistent with your preprocessing. Tokenizing the stop words generated tokens ['ha', 'u', 'wa'] not in stop_words.
  warnings.warn(


KeyboardInterrupt: Interrupted by user

# Build a Chatbot with Neural Network 

We've discovered how to build a chatbot with cosine similarity. Now, let's explore how we might build one with neural network!

We will create our training data, train a neural network with them, then use the trained model to make our chatbot. 

First, we will install required libraries. Uncomment the few blocks below only if you do not have the libraries installed. 

In [13]:
#!pip install numpy scipy
#!pip install scikit-learn
#!pip install pillow
#!pip install h5py

In [14]:
#!pip install tensorflow

In [15]:
#!pip install tensorflow-gpu

In [16]:
#!pip install keras

# 1. Install Libraries

Firstly, we will install libraries needed for this neural network powered chatbot. 
Keras is a machine learning library which utilizes tensorflow (another lower level machine learning library) at the backend. This makes it easier for us to deploy deep neural network for this purpose. 

In [3]:
from keras.models import Sequential
from keras.losses import categorical_crossentropy
from keras.optimizers import SGD
from keras.layers import Dense
 
from numpy import argmax
import numpy as np
import re

# 2. Input training data

We will first include the following training data for our chatbot:
1. X represent the different possible inputs that users might enter
2. Y represent the intent of the inputs

In [4]:
X = ['Hi',
     'Hello',
     'How are you?',
     'I am making',
     'making',
     'working',
     'studying',
     'see you later',
     'bye',
     'goodbye']

In [5]:
print(len(X))

10


In [6]:
Y = ['greeting',
     'greeting',
     'greeting',
     'busy',
     'busy',
     'busy',
     'busy',
     'bye',
     'bye',
     'bye']

In [7]:
print(len(Y))

10


Notice that there are several different sentences that have similar intent. Here, we are only having 3 intents, but you can add as many as you want for your project!

This is the way our chatbot will work:
1. From the input sentence, we will identify the intent using our trained AI model.
2. For each intent, we have a prepared response. 

For example, if we identify that the intent of the input is for a greeting, we might ask the chatbot to reply with a greeting as well, something like 'hi' or 'how are you doing?'

We will use machine learning to create a model that can classify input sentence into different intents. 
We make it as follows:

1. We create a training data (X and Y above) which contains a list of sentences and their intents.
2. Use the training data to train a classifier. 
3. Vectorize input sentences and use classifier to determine intent. 

# 3. Text processing

As usual, we will start with text processing. Do you remember the process?

## 3.1 Remove non alphanumeric characters

In [8]:
def remove_non_alpha_numeric_characters(sentence):
    new_sentence = ''
    for alphabet in sentence:
        if alphabet.isalpha() or alphabet == ' ':
            new_sentence += alphabet
    return new_sentence

In [9]:
def preprocess_data(X):
    X = [data_point.lower() for data_point in X]
    X = [remove_non_alpha_numeric_characters(
        sentence) for sentence in X]
    X = [data_point.strip() for data_point in X]
    X = [re.sub(' +', ' ',
                data_point) for data_point in X]
    return X

In [10]:
X = preprocess_data(X)

vocabulary = set()
for data_point in X:
    for word in data_point.split(' '):
        vocabulary.add(word)

vocabulary = list(vocabulary)

## Create document vectors

In [11]:
X_encoded = []

def encode_sentence(sentence):
    sentence = preprocess_data([sentence])[0]
    sentence_encoded = [0] * len(vocabulary)
    for i in range(len(vocabulary)):
        if vocabulary[i] in sentence.split(' '):
            sentence_encoded[i] = 1
    return sentence_encoded

X_encoded = [encode_sentence(sentence) for sentence in X]

In [12]:
classes = list(set(Y))

Y_encoded = []
for data_point in Y:
    data_point_encoded = [0] * len(classes)
    for i in range(len(classes)):
        if classes[i] == data_point:
            data_point_encoded[i] = 1
    Y_encoded.append(data_point_encoded)

# 4. Create training data and test data

In [13]:
X_train = X_encoded
y_train = Y_encoded
X_test = X_encoded
y_test = Y_encoded

Print and check the data you are using for training and test data

In [14]:
print (y_test)

[[1, 0, 0], [1, 0, 0], [1, 0, 0], [0, 0, 1], [0, 0, 1], [0, 0, 1], [0, 0, 1], [0, 1, 0], [0, 1, 0], [0, 1, 0]]


In [15]:
print(X_train)

[[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0], [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0], [0, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0], [0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0], [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1], [1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]]


In [16]:
print(len(X_train))

10


In [17]:
y_train

[[1, 0, 0],
 [1, 0, 0],
 [1, 0, 0],
 [0, 0, 1],
 [0, 0, 1],
 [0, 0, 1],
 [0, 0, 1],
 [0, 1, 0],
 [0, 1, 0],
 [0, 1, 0]]

What does y_train represent? Do you understand the array shown above?

# 5. Model training

Now we will use the training data to train our neural network.

In [18]:
model = Sequential()
model.add(Dense(units=64, activation='sigmoid',
                input_dim=len(X_train[0])))
model.add(Dense(units=len(y_train[0]), activation='softmax'))
model.summary()

/home/nate/venvs/nlp/lib/python3.13/site-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-03-28 15:05:17.631940: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │           960 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 3)              │           195 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,155 (4.51 KB)

 Trainable params: 1,155 (4.51 KB)

 Non-trainable params: 0 (0.00 B)

In [19]:
import numpy as np
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import SGD

num_features = np.array(X_train).shape[1]
num_classes = np.array(y_train).shape[1]

#model = Sequential([
#    Input(shape=(num_features,)),
#    Dense(64, activation="sigmoid"),
#    Dense(num_classes, activation="softmax")
#])

model.compile(
    loss="categorical_crossentropy",
    optimizer=SGD(learning_rate=0.01, momentum=0.9, nesterov=True),
    metrics=["accuracy"]
)

model.fit(np.array(X_train), np.array(y_train), epochs=100, batch_size=16)



Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 416ms/step - accuracy: 0.4000 - loss: 1.1949
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.4000 - loss: 1.1829
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.4000 - loss: 1.1677
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.4000 - loss: 1.1517
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 32ms/step - accuracy: 0.4000 - loss: 1.1367
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.4000 - loss: 1.1245
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.4000 - loss: 1.1159
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.4000 - loss: 1.1111
Epoch 9/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - accuracy: 0.4000 - loss: 1.1093
Epoch 10/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.4000 - loss: 1.1096
Epoch 11/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.3000 - loss: 1.1107
Epoch 12/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.2000 - l

## List down predictions

In [20]:
predictions = [argmax(pred) for pred in model.predict(np.array(X_test))]

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step


# Model Evaluation

Let's evaluate our model now. We will compare the prediction made by the model and our test data:

In [21]:
correct = 0
for i in range(len(predictions)):
    if predictions[i] == argmax(y_test[i]):
        correct += 1

print ("Correct:", correct)
print ("Total:", len(predictions))

Correct: 6
Total: 10


# Testing the chatbot

Let's test the chatbot now! We will input a sentence, and then see what class is predicted by the neural network:

In [22]:
while True:
    print ("Enter a sentence")
    sentence = input()
    prediction= model.predict(np.array([encode_sentence(sentence)]))
    print (classes[argmax(prediction)])

Enter a sentence


 Hello


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
busy
Enter a sentence


KeyboardInterrupt: Interrupted by user

Realize that you can't stop the chatbot? You'll have to add the exit command later (see the previous notebook to find out how to do it. 

For now, simply press the stop button (interrupt button) above to stop the chatbot. 

Try it! press the stop button, and try typing something onto the box. 

# Challenge

We have successfully use neural network to map our input to conversation intent. 
Your challenge is to link the conversation intent to a particular response that the chatbot will say. 
For example, if the conversation intent is 'greeting', get your chatbot to say a greeting as well!

In [44]:
while True:
    print ("Enter a sentence")
    sentence = input()
    prediction= model.predict(np.array([encode_sentence(sentence)]))
    #print (classes[argmax(prediction)])
    if (classes[argmax(prediction)] == 'bye'):
        print ('Goodbye! See you next time!')
    elif (classes[argmax(prediction)] == 'busy'):
        print('ah... all the best!')
    elif (classes[argmax(prediction)] == 'greeting'):
        print ('Hi!')

Enter a sentence


KeyboardInterrupt: Interrupted by user

### Great job! You've successfully created a simple chatbot with neural network! How might you improve the chatbot?
You can improve the chatbot by:
- Adding more training data
- Adding more intent
- Focusing on a particular topic and train the chatbot with many training data in that topic

### Resource:
https://blog.eduonix.com/internet-of-things/simple-nlp-based-chatbot-python/

## Submission Instructions
    Run all code cells successfully
    
    Complete all markdown responses
    
    Export the notebook as HTML:
    
    File → Download as → HTML (.html)
    
    Submit the HTML file in Canvas by the posted deadline

## Setup (Run First)
If ChatterBot is not already installed, install it once in your environment.

In [1]:
# Install ChatterBot if needed (uncomment and run once)
#!pip install chatterbot chatterbot-corpus

## Part 1: Set Up and Train the Chatbot (30 points)
In this section, you will create and train a ChatterBot instance using the provided corpus.

Questions (Answer in Markdown Cells)

    1. Did the chatbot respond sensibly to “What is a chatbot?”? Quote the response.

    2. Try two additional queries from the corpus. What responses did you receive?


In [2]:
from pathlib import Path

from chatterbot import ChatBot
from chatterbot.trainers import ListTrainer
from chatterbot.logic import LogicAdapter
from chatterbot.conversation import Statement

In [3]:
# Step 1: Imports and Custom Weather Adapter
class WeatherAdapter(LogicAdapter):
    """
    Custom logic adapter:
    If the input contains the word 'weather',
    return a fixed weather response.
    """

    def __init__(self, chatbot, **kwargs):
        super().__init__(chatbot, **kwargs)

    def can_process(self, statement):
        return "weather" in statement.text.lower()

    def process(self, input_statement, additional_response_selection_parameters=None):
        response = Statement(text="The weather is sunny today!")
        response.confidence = 1.0
        return response

In [4]:
# Step 2: Load the Corpus and Train the Bot
def load_corpus_lines(filepath="chatbots.txt"):
    path = Path(filepath)
    if not path.exists():
        raise FileNotFoundError(f"Corpus file not found: {filepath}")

    lines = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                lines.append(line)
    return lines


In [5]:
bot = ChatBot(
    "MyChatBot",
    logic_adapters=[
        {"import_path": "__main__.WeatherAdapter"},
        {"import_path": "chatterbot.logic.BestMatch"},
    ],
)

corpus = load_corpus_lines("/home/nate/NextCloud/Roam/Classes/NLP/week_11/chatbots.txt")
trainer = ListTrainer(bot)
trainer.train(corpus)

print("Training complete.")

List Trainer: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 67/67 [00:00<00:00, 20696.60it/s]

Training complete.


In [7]:
# Step 3: Test Basic Responses
response = bot.get_response("Who is Alan Turing?")
print("Bot:", response)

Bot: In 1950, Alan Turing's famous article "Computing Machinery and Intelligence" was published,[6] which proposed what is now called the Turing test as a criterion of intelligence. This criterion depends on the ability of a computer program to impersonate a human in a real-time written conversation with a human judge to the extent that the judge is unable to distinguish reliably—on the basis of the conversational content alone—between the program and a real human. The notoriety of Turing's proposed test stimulated great interest in Joseph Weizenbaum's program ELIZA, published in 1966, which seemed to be able to fool users into believing that they were conversing with a real human. However Weizenbaum himself did not claim that ELIZA was genuinely intelligent, and the introduction to his paper presented it more as a debunking exercise:


1. The basic question of "What is a chatbot?" give an actual sensible response, being of
   "Most chatbots are accessed on-line via website popups or through virtual assistants. They can be classified into usage categories that include: commerce (e-commerce via chat), education, entertainment, finance, health, news, and productivity"
   
2. Given two questions I've derived from the chatbot.txt courpus I've gotten,
   - "What is the history behind chatbots like ELIZA"?
     Bot: The creation and implementation of chatbots is still a developing area, heavily related to artificial intelligence and machine learning, so the provided solutions, while possessing obvious advantages, have some important limitations in terms of functionalities and use cases. However, this is changing over time.     

   - "Who is Alan Turing?"
     Bot: In 1950, Alan Turing's famous article "Computing Machinery and Intelligence" was published,[6] which proposed what is now called the Turing test as a criterion of intelligence. This criterion depends on the ability of a computer program to impersonate a human in a real-time written conversation with a human judge to the extent that the judge is unable to distinguish reliably—on the basis of the conversational content alone—between the program and a real human. The notoriety of Turing's proposed test stimulated great interest in Joseph Weizenbaum's program ELIZA, published in 1966, which seemed to be able to fool users into believing that they were conversing with a real human. However Weizenbaum himself did not claim that ELIZA was genuinely intelligent, and the introduction to his paper presented it more as a debunking exercise:  
 


## Part 3: Build a Conversation Loop (20 points)
To avoid issues with notebook execution and grading, use the scripted conversation loop below.

Questions (Answer in Markdown Cells)

    1. Show a short conversation transcript (5–10 exchanges).

    2. What worked well during the conversation?

    3. What failed or produced poor responses?

In [11]:
scripted_conversation = [
    "Hello",
    "What is a chatbot?",
    "What's the weather today?",
    "Tell me something interesting.",
    "exit" 
]

print("Conversation transcript:\n")

for user_input in scripted_conversation:
    print(f"User: {user_input}")
    if user_input.lower() == "exit":
        print("Bot: Goodbye!")
        break
    response = bot.get_response(user_input)
    print(f"Bot: {response}\n")

Conversation transcript:

User: Hello
Bot: IBM's Watson computer has been used as the basis for chatbot-based educational toys for companies such as CogniToys[64] intended to interact with children for educational purposes.[68]

User: What is a chatbot?
Bot: See also: Artificial intelligence in healthcare

User: What's the weather today?
Bot: The weather is sunny today!

User: Tell me something interesting.
Bot: Malicious chatbots are frequently used to fill chat rooms with spam and advertisements, by mimicking human behavior and conversations or to entice people into revealing personal information, such as bank account numbers. They were commonly found on Yahoo! Messenger, Windows Live Messenger, AOL Instant Messenger and other instant messaging protocols. There has also been a published report of a chatbot used in a fake personal ad on a dating service's website.[69]

User: exit
Bot: Goodbye!


2. Questions which are asked to find something of interest or that which invoked the WeatherAdapter class set of functions work well. The former seemed to work as such due to the corpus being of a topic matter that could be of inetrest to a reader and of the programtica automation of answer related to the weather.   
3. Asking about a chatbot had produced non-conversiontal results while also saying hello resulted in a factoid of IBM's Watson computer. The latter could just be the corpus, but the former seems to be odd when the corpus itself is full of information about chatbots; what makes the former more odd is that there is such a default response for a input like "Hello". 



## Part 4: Comparison to Other Chatbot Approaches (20 points)
Review the chatbot approaches demonstrated earlier in the course (rule-based and neural chatbots).

Comparison (Complete in Markdown)

Reflection Questions (3–5 sentences each)

    1. How does ChatterBot differ from rule-based and neural chatbot approaches?

    2. If you were designing a customer support chatbot for a business, which approach would you choose and why?

1. In this context, the greatest difference between that of rule-based & nerual net from that of Chattbot seems generally to be the scope of the discussion lead by that of classification. Chatterbot by taking in a large corpus of information is better posed with it's pipeline to produce a more natural conversation then that of the rule-based or nerual net.

   There is some difference between that of the nerual net & rule-based wherein the latter is constructred of feedforward two layer neural net with a softmax function for classification on it's head. The rule based is programmatic classification via if/else statements.  

2. To start with it would be a no-brainer for me to use something like Chatterbot, but I would focus on it's ablity to save statements in order to improve it's responses over time if repeated issues start to emerage. I don't fully think that a chatbot based of chatterbot would be ideal for all sub-contexts of the use of such chatbot in this context: it think denedends on what it's supposed to do.

   I think what could be another solution is to take a look towards currently evoloving architectures which use modules like that of symbolic components for checking the output of LLM-based output. Such router-based patterns within the overall architecture could lend to a more creative use of such model types when dealing with complex, non-linear customer serivce request. 



### Rule-Based Chatbots

Pros:
* Predictability and Reliability: Since they follow a strict if-then logic, the output is always consistent and follows brand guidelines.
* Low Computational Cost: They require very little processing power and can run on minimal hardware compared to neural models.
* Ease of Debugging: Because the logic path is visible, it is easy to find and fix the specific rule causing an error.

Cons:
* Scalability Issues: Manually writing rules for every possible user query becomes impossible as the scope of the bot grows.
* Zero Flexibility: They cannot understand typos, slang, or context outside of their pre-defined keywords.
* Poor User Experience: Users often feel trapped in a phone tree loop if their specific problem is not a menu option.

Typical Use Cases:
* Basic FAQ automation.
* Simple data collection like gathering a name and email for a lead.
* Order tracking and simple transactional updates.

### Neural Chatbots

Pros:
* Contextual Awareness: They can maintain a memory of the conversation, understanding pronouns and follow-up questions.
* Natural Language Handling: They excel at understanding intent even if the user uses unconventional phrasing or makes typos.
* Continuous Improvement: These models can be fine-tuned on new data to get better at specific tasks over time.

Cons:
* Hallucinations: They can confidently state facts that are entirely fabricated because they are predicting the next word, not checking a database.
* High Resource Demand: Training and running these models requires significant GPU power and high latency compared to rules.
* Black Box Nature: It can be difficult to explain why a model gave a specific answer, making them harder to steer with 100% precision.

Typical Use Cases:
* Creative writing and brainstorming assistants.
* High-level customer support capable of nuanced problem-solving.
* Personal virtual assistants like sophisticated LLM agents.

### ChatterBot (Retrieval-Based)

Pros:
* Safe Responses: Since it selects from a library of pre-existing human responses, it will not invent language or use inappropriate grammar.
* Easy to Seed: You can jumpstart the bot by feeding it a transcript of previous chat logs or help desk tickets.
* Language Agnostic: Because it matches patterns rather than understanding deep semantics, it can be adapted to different languages relatively easily.

Cons:
* Repetitive: If the database is small, the bot will give the exact same response to many different types of questions.
* Lacks Creativity: It cannot synthesize a new answer; if the answer is not in the database, the bot is stuck.
* Database Maintenance: As information changes, you must manually find and update every relevant entry in the response library.

Typical Use Cases:
* Technical support where specific, verified instructions are required.
* Community management bots that echo official answers to common questions.
* Simple social bots or chatter applications meant for entertainment.